# Classifier based on the Title

## Dependencies

In [3]:
import numpy as np
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.initializers import Constant
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from collections import Counter



## Loading Data & preprocessing

In [4]:
tf_class_data = pd.read_csv("../data/title_class_dataset.csv")
# --- Setup ---
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


tf_stop_words = set(stopwords.words('english'))
tf_lemmatizer = WordNetLemmatizer()

# --- Preprocessing Function ---
def preprocess_text(text):
    if not(isinstance(text, str)):
        return ""
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in tf_stop_words]
    tokens = [tf_lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)


tf_class_data['title'] = tf_class_data['title'].apply(preprocess_text)


# --- Tokenization ---
tf_tokenizer = Tokenizer(num_words=10000, oov_token='<UNK>')
tf_tokenizer.fit_on_texts(tf_class_data['title'])
tf_sequences = tf_tokenizer.texts_to_sequences(tf_class_data['title'])
print(len(tf_tokenizer.word_index))

tf_word_index = tf_tokenizer.word_index

tf_max_seq_length = max(len(seq) for seq in tf_sequences)
X = pad_sequences(tf_sequences, maxlen=tf_max_seq_length, padding='post')
y = tf_class_data['label'].astype(int).values

# --- Split ---
tf_X_train, tf_X_test, tf_y_train, tf_y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Load GloVe ---
tf_embedding_index = {}
with open("../glove.6B/glove.6B.100d.txt", 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        tf_embedding_index[word] = vector

tf_embedding_dim = 100
tf_embedding_matrix = np.zeros((len(tf_word_index) + 1, tf_embedding_dim))
for word, i in tf_word_index.items():
    embedding_vector = tf_embedding_index.get(word)
    if embedding_vector is not None:
        tf_embedding_matrix[i] = embedding_vector
    else:
        tf_embedding_matrix[i] = np.random.normal(scale=0.6, size=(tf_embedding_dim,))

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/carlosdesousa/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/carlosdesousa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/carlosdesousa/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


32936


## Model Definition

In [5]:
tf_model = Sequential([
    Embedding(input_dim=len(tf_word_index)+1,
              output_dim=tf_embedding_dim,
              embeddings_initializer=Constant(tf_embedding_matrix),
              input_length=tf_max_seq_length,
              trainable=False),
    LSTM(128),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

/opt/anaconda3/envs/app-ml/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## Training

In [6]:
tf_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
tf_model.summary()

# --- Train ---
tf_model.fit(tf_X_train, tf_y_train, epochs=10, batch_size=32, validation_split=0.2)

# --- Evaluate ---
loss, accuracy = tf_model.evaluate(tf_X_test, tf_y_test)
print(f"Test Accuracy: {accuracy:.4f}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1432/1432 ━━━━━━━━━━━━━━━━━━━━ 32s 21ms/step - accuracy: 0.7982 - loss: 0.4318 - val_accuracy: 0.8784 - val_loss: 0.2875
Epoch 2/10
1432/1432 ━━━━━━━━━━━━━━━━━━━━ 28s 20ms/step - accuracy: 0.8804 - loss: 0.2826 - val_accuracy: 0.8938 - val_loss: 0.2559
Epoch 3/10
1432/1432 ━━━━━━━━━━━━━━━━━━━━ 28s 19ms/step - accuracy: 0.9000 - loss: 0.2444 - val_accuracy: 0.8969 - val_loss: 0.2551
Epoch 4/10
1432/1432 ━━━━━━━━━━━━━━━━━━━━ 29s 20ms/step - accuracy: 0.9131 - loss: 0.2140 - val_accuracy: 0.9025 - val_loss: 0.2448
Epoch 5/10
1432/1432 ━━━━━━━━━━━━━━━━━━━━ 157s 110ms/step - accuracy: 0.9222 - loss: 0.1935 - val_accuracy: 0.9010 - val_loss: 0.2547
Epoch 6/10
1432/1432 ━━━━━━━━━━━━━━━━━━━━ 27s 19ms/step - accuracy: 0.9338 - loss: 0.1687 - val_accuracy: 0.9076 - val_loss: 0.2638
Epoch 7/10
1432/1432 ━━━━━━━━━━━━━━━━━━━━ 29s 20ms/step - accuracy: 0.9429 - loss: 0.1486 - val_accuracy: 0.9087 - val_loss: 0.2537
Epoch 8/10
1432/1432 ━━━━━━━━━━━━━━━━━━━━ 28s 20ms/step - accuracy: 0.9487